# Fraud Detection Model
This notebook outlines data cleaning, modeling, and evaluation steps for predicting fraudulent transactions.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns


## 1. Load data
The dataset can be downloaded from Google Drive. The code below assumes the file is available as `transactions.csv`.

In [ ]:
# Replace with actual path or download logic
df = pd.read_csv('transactions.csv')
df.head()

## 2. Data Cleaning
Handle missing values, remove outliers, and check for multicollinearity.

In [ ]:
# Missing values
df = df.dropna()

# Outlier removal using IQR
numeric_cols = df.select_dtypes(include=['float64','int64']).columns
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1
df = df[~((df[numeric_cols] < (Q1 - 1.5 * IQR)) | (df[numeric_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]

# Multicollinearity via VIF
X_vif = df[numeric_cols].drop('fraud', axis=1, errors='ignore')
vif = pd.DataFrame()
vif['feature'] = X_vif.columns
vif['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif

## 3. Feature Selection and Split
Select relevant variables and create train/validation sets.

In [ ]:
features = [col for col in df.columns if col != 'fraud']
X = df[features]
y = df['fraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 4. Model Training
Train a random forest model.

In [ ]:
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

## 5. Evaluation
Evaluate the model using confusion matrix, classification report, and ROC-AUC.

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()
y_prob = model.predict_proba(X_test)[:,1]
roc_auc = roc_auc_score(y_test, y_prob)
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.show()
print('ROC-AUC:', roc_auc)

## 6. Feature Importance
Identify key predictors of fraud.

In [ ]:
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
importances.head(10)

## 7. Interpretation and Next Steps
- Describe how variables were selected (based on VIF and domain knowledge).
- Discuss whether key factors make sense.
- Recommend infrastructure updates such as real-time monitoring, anomaly detection, and periodic model retraining.
- To verify effectiveness, monitor fraud rates before and after implementation and use A/B testing on flagged transactions.